# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata object DIRECTLY
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll use dataset methods to list the record sets, fields, and columns defined in the schema. All entities will be referenced by their `@id`.

In [ ]:
# List available record sets and their @ids
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"  Name: {rs.name} | @id: {rs.id}")

# List fields for each record set
for rs in record_sets:
    print(f"\nFields in Record Set '{rs.name}' (@id={rs.id}):")
    for field in rs.fields:
        print(f"    Field: {field.name} | @id: {field.id} | Data Type: {field.data_type}")

# List columns for each record set (if present)
for rs in record_sets:
    if hasattr(rs, 'columns') and rs.columns:
        print(f"\nColumns in Record Set '{rs.name}' (@id={rs.id}):")
        for col in rs.columns:
            print(f"    Column: {col.name} | @id: {col.id} | Data Type: {col.data_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**Note:** Use the record set and field `@id` values obtained above.

In [ ]:
# Prepare a list of record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

# Dictionary to store DataFrames for each record set
dfs = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)

# Show columns for the first record set
first_rs_id = record_set_ids[0]
print(f"Columns in the first record set (@id={first_rs_id}):")
print(dfs[first_rs_id].columns.tolist())

# Show head of the first record set DataFrame
dfs[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping data by key attributes.

We'll demonstrate numeric filtering, normalization, and grouping by categorical column, referencing all fields by their `@id`.

In [ ]:
# Choose the record set to analyze (example: use first record set)
record_set_id = record_set_ids[0]
df = dfs[record_set_id]

# Identify numeric and categorical fields by @id
record_set = next(rs for rs in dataset.record_sets if rs.id == record_set_id)
numeric_fields = [field.id for field in record_set.fields if field.data_type in ['Integer', 'Float', 'Number']]
categorical_fields = [field.id for field in record_set.fields if field.data_type == 'Text']

# Example: Take the first available numeric field and a categorical field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = None

if categorical_fields:
    group_field_id = categorical_fields[0]
else:
    group_field_id = None

# Filter for values greater than threshold if numeric field exists
if numeric_field_id and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This section uses basic histograms and bar charts for numeric/categorical fields, always referencing by `@id`.

In [ ]:
# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

# Example histogram of numeric field, barplot for categorical group
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

if group_field_id and group_field_id in df.columns and numeric_field_id:
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=df, ci=None)
    plt.title(f"Mean {numeric_field_id} per {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- This notebook demonstrated how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.
- We referenced all entities (record sets, fields, columns) by their `@id` for consistency and reproducibility.
- Using `mlcroissant`, you can extract data for further statistical or machine learning analysis, and visualize relationships between key clinical and molecular attributes.